In [1]:
# Import von Bibliotheken
from datasets import load_dataset
!pip install pypdf
import urllib.request
import io,re
import json
from pypdf import PdfReader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
!pip install textstat   # Lesbarkeitindizes Wiener Sachtextformel & Flesch-Amstad-Index
import textstat

  Using cached pypdf-6.15.0-py3-none-any.whl.metadata (7.5 kB)
Using cached pypdf-6.15.0-py3-none-any.whl (378 kB)
  Using cached textstat-0.7.13-py3-none-any.whl.metadata (15 kB)
  Using cached pyphen-0.17.2-py3-none-any.whl.metadata (3.2 kB)
Using cached textstat-0.7.13-py3-none-any.whl (177 kB)
Using cached pyphen-0.17.2-py3-none-any.whl (2.1 MB)


**1. Teil: Wahlprogramme**

In [2]:
# Einlesen der Langwahlprogramme (aus JSON-Format in ein Wörterbuch)

# für CDU/CSU
with open("cdu-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    CDU_text_json = json.load(datei)

# für SPD
with open("spd-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    SPD_text_json = json.load(datei)

# für Die Linke
with open("linke-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    Linke_text_json = json.load(datei)

# für die AfD
with open("afd-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    AfD_text_json = json.load(datei)

# für Bündnis 90/Grüne
with open("gruene-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    Grüne_text_json = json.load(datei)

# für FDP
with open("fdp-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    FDP_text_json = json.load(datei)

In [3]:
# Überführen aller 'eigentlichen' Texte in eine Liste pro Partei, d.h. ohne Kapitelüberschriften

# Suchfunktion in den "Partei-Wörterbüchern"

def finde_gefilterte_texte(struktur):
    ergebnisse = []
    
    # Fall A: Wenn das aktuelle Element ein Dictionary ist
    if isinstance(struktur, dict):
        # Typ/Art auslesen (liefert None, wenn der Schlüssel fehlt)
        aktueller_typ = struktur.get('type') or struktur.get('art')            # nur Suchen in diesen Elementen
        
        # Prüfen, ob 'text' existiert und der gefundene Typ gültig ist
        if 'text' in struktur and aktueller_typ in ['paragraph', 'bullet', 'absatz']:   # nur Texte überführen mit diesem Typ
            ergebnisse.append(struktur['text'])
        
        # Tiefer in alle Werte schauen für eventuelle Verschachtelungen
        for wert in struktur.values():
            ergebnisse.extend(finde_gefilterte_texte(wert))
                
    # Fall B: Wenn das aktuelle Element eine Liste ist
    elif isinstance(struktur, list):
        for element in struktur:
            ergebnisse.extend(finde_gefilterte_texte(element))
            
    return ergebnisse

CDU_text = finde_gefilterte_texte(CDU_text_json)
SPD_text = finde_gefilterte_texte(SPD_text_json)
Linke_text = finde_gefilterte_texte(Linke_text_json)
Grüne_text = finde_gefilterte_texte(Grüne_text_json)
AfD_text = finde_gefilterte_texte(AfD_text_json)
FDP_text = finde_gefilterte_texte(FDP_text_json)

In [4]:
# Überführen aller 'eigentlichen' Texte in einen String pro Partei, d.h. ohne Kapitelüberschriften

# für CDU/CSU
CDU_text_str = [text if text.endswith('.') else text + '.' for text in CDU_text]   # ein "Punkt" wird gesetzt bei Bulletpoints
CDU_text = " ".join(CDU_text_str)

# für SPD
SPD_text_str = [text if text.endswith('.') else text + '.' for text in SPD_text]
SPD_text = " ".join(SPD_text_str)

# für Die Linke
Linke_text_str = [text if text.endswith('.') else text + '.' for text in Linke_text]
Linke_text = " ".join(Linke_text_str)

# für die AfD
AfD_text_str = [text if text.endswith('.') else text + '.' for text in AfD_text]
AfD_text = " ".join(AfD_text_str)

# für Bündnis 90/Grüne
Grüne_text_str = [text if text.endswith('.') else text + '.' for text in Grüne_text]
Grüne_text = " ".join(Grüne_text_str)

# für FDP
FDP_text_str = [text if text.endswith('.') else text + '.' for text in FDP_text]
FDP_text = " ".join(FDP_text_str)

In [5]:
# Bereinigung der Texte - TEIL 1 -> um Sonderzeichen usw., siehe Kommentar neben entsprechendem reg-Ausdruck

def str_bereinigen(i):
    i = re.sub(r'-\s+', '', i)                               # Ersetzt den Bindestrich gefolgt von einem oder mehreren Leerzeichen durch nichts
    i = re.sub(r'[^\w\s\.!\?€,,:\(\)\-%]', '', i)            # Sonderzeichen-Filter, löscht alles, was nicht ausdrücklich erhalten bleiben soll
    i = re.sub(r'([A-Za-zÄÖÜäöüß])\1{3,}', r'\1\1\1', i)     # Begrenzung von extremen Wiederholungen
    i = re.sub(r'\s+', ' ', i).strip()                       # Normalisierung von Whitespaces (ersetzt durch ein Leerzeichen)
    return i
    
CDU_text = str_bereinigen(CDU_text)
SPD_text = str_bereinigen(SPD_text)
Grüne_text = str_bereinigen(Grüne_text)
AfD_text = str_bereinigen(AfD_text)
Linke_text = str_bereinigen(Linke_text)
FDP_text = str_bereinigen(FDP_text)


In [6]:
# Bereinigen der zuvor bereinigten Texte - TEIL 2 => Umformen von Abkürzungen in Text um Eindruck von einem Satzende zu vermeiden

def bereinige_abkuerzungen(text):
    # Dictionary mit den Top 20 Ersetzungen (Regex-Muster als Key)
    ersetzungen = {
        r'\bz\.\s*B\.': 'zum Beispiel',   # r'\bz\.\s*B\.\b': 'zum Beispiel',
        r'\bu\.\s*a\.': 'unter anderem',
        r'\bd\.\s*h\.': 'das heißt',
        r'\bbzw\.': 'beziehungsweise',
        r'\bbzw': 'beziehungsweise',   # Variante ohne Punkt
        r'\bsog\.': 'sogenannte',
        r'\bca\.': 'circa',
        r'\bevtl\.': 'eventuell',
        r'\binkl\.': 'inklusive',
        r'\betc\.': 'et cetera',
        r'\bvgl\.': 'vergleiche',
        r'\bs\.': 'siehe',
        r'\bu\.\s*v\.\s*m\.': 'und vieles mehr',
        r'\bu\.\s*ä\.': 'und ähnliche',
        r'\bggf\.': 'gegebenenfalls',
        r'\bzzgl\.': 'zuzüglich',
        r'\bebd\.': 'ebenda',
        r'\bo\.\s*g\.': 'oben genannte',
        r'\bu\.\s*g\.': 'unten genannte',
        r'\bi\.\s*d\.\s*R\.': 'in der Regel',
        r'\bv\.\s*a\.': 'vor allem',

        # 2. Titel & Personen (Verhindern Satzabbruch mitten im Fluss)
        r'\bDr\.': 'Doktor',
        r'\bProf\.': 'Professor',
        r'\bFr\.': 'Frau',
        r'\bHr\.': 'Herr',

        # 3. Wortlaengen- & Silben-Verfaelscher
        r'\bbspw\.': 'beispielsweise',
        r'\bbspw': 'beispielsweise',   # Variante ohne Punkt
        r'\bbsp\.': 'Beispiel',
        r'\bJh\.': 'Jahrhundert',
        r'\bMio\.': 'Millionen',
        r'\bMrd\.': 'Milliarden',
        r'\bbetr\.': 'betreffend',
        r'\bbezgl\.': 'bezüglich',
        r'\bvs\.': 'versus'
    }

    # Text Schritt für Schritt bereinigen
    for muster, ersetzung in ersetzungen.items():
        # flags=re.IGNORECASE sorgt dafür, dass auch "Z.B." oder "Bzw." gefunden werden
        text = re.sub(muster, ersetzung, text, flags=re.IGNORECASE)

    return text

CDU_text = bereinige_abkuerzungen(CDU_text)
SPD_text = bereinige_abkuerzungen(SPD_text)
Grüne_text = bereinige_abkuerzungen(Grüne_text)
AfD_text = bereinige_abkuerzungen(AfD_text)
Linke_text = bereinige_abkuerzungen(Linke_text)
FDP_text = bereinige_abkuerzungen(FDP_text)

In [7]:
# Sprache auf Deutsch stellen und Werte ermitteln
textstat.set_lang("de")
CDU_WSF = textstat.wiener_sachtextformel(CDU_text, 1)   # '1' steht für 1. Wiener Sachtextformel
SPD_WSF = textstat.wiener_sachtextformel(SPD_text, 1)
Linke_WSF = textstat.wiener_sachtextformel(Linke_text, 1)
AfD_WSF = textstat.wiener_sachtextformel(AfD_text, 1)
Grüne_WSF = textstat.wiener_sachtextformel(Grüne_text, 1)
FDP_WSF = textstat.wiener_sachtextformel(FDP_text, 1)

textstat.set_lang("de")
CDU_FRE = textstat.flesch_reading_ease(CDU_text)
SPD_FRE = textstat.flesch_reading_ease(SPD_text)
Linke_FRE = textstat.flesch_reading_ease(Linke_text)
AfD_FRE = textstat.flesch_reading_ease(AfD_text)
Grüne_FRE = textstat.flesch_reading_ease(Grüne_text)
FDP_FRE = textstat.flesch_reading_ease(FDP_text)

In [8]:
# PandaDF-Erstellung
wsf_series = pd.Series({
    "CDU": CDU_WSF,
    "SPD": SPD_WSF,
    "Linke": Linke_WSF,
    "AfD": AfD_WSF,
    "Grüne": Grüne_WSF,
    "FDP": FDP_WSF}).round(1)

FRE_series = pd.Series({
    "CDU": CDU_FRE,
    "SPD": SPD_FRE,
    "Linke": Linke_FRE,
    "AfD": AfD_FRE,
    "Grüne": Grüne_FRE,
    "FDP": FDP_FRE}).round(1)

df_wahlprogramm = pd.DataFrame(
    [wsf_series, FRE_series],
    index=["1. WSTF", "Flesch-Amstad-Index"]
)

df_wahlprogramm.columns = ["CDU/CSU", "SPD", "Linke", "AfD", "Grüne", "FDP"]
df_wahlprogramm["⌀"] = df_wahlprogramm.mean(axis=1).round(1)
cols = ["CDU/CSU", "SPD", "AfD", "Grüne", "FDP"]
#df_wahlprogramm["⌀ ohne Linke"] = df_wahlprogramm[cols].mean(axis=1).round(1)
df_wahlprogramm

,CDU/CSU,SPD,Linke,AfD,Grüne,FDP,⌀
1. WSTF,11.5,12.0,11.9,13.2,12.5,12.3,12.2
Flesch-Amstad-Index,35.5,31.8,32.6,27.9,29.1,30.6,31.2


In [9]:
# in LaTex überführen

latex_code = df_wahlprogramm.to_latex(index=False, caption='Meine Übersicht', label='tab:meine_tabelle')
with open('tabelle_wstf_fre.tex', 'w') as f:
    f.write(latex_code)

**2. Teil: Bundestagsreden**

In [10]:
import json
import urllib.request, urllib.parse, urllib.error
stammdaten = ("MDB_STAMMDATEN.XML")  # mit den Parteizugehörigkeiten
from bs4 import BeautifulSoup
with open(stammdaten, "r", encoding="utf-8") as f:
  	soup2 = BeautifulSoup(f, "xml")

redner_id_mdb = dict()
for mdb in soup2.find_all("MDB"):
	id_tag = mdb.find("ID")
	party_tag = mdb.find("PARTEI_KURZ")
	mdb_id = id_tag.text if id_tag else None
	party = party_tag.text if party_tag else None
	redner_id_mdb[mdb_id] = party

# da wo keine Parteizugehörigkeit aus den Stammdaten ersichtlich, manuelles Hinzufügen diverser Nummern
redner_id_mdb["11002735"]  # MERZ
redner_id_mdb["999990151"] = "SPD"
redner_id_mdb["999990133"] = "SPD"
redner_id_mdb["999990074"] = "SPD"
redner_id_mdb["11005217 999990074"] = "SPD"
redner_id_mdb["999990119"] = "SPD"
redner_id_mdb["999990149"] = "SPD"
redner_id_mdb["999990080"] = "parteilos"
redner_id_mdb["999990142"] = "parteilos"
redner_id_mdb["999990154"] = "parteilos"
redner_id_mdb["999990152"] = "CDU"
redner_id_mdb["999990153"] = "CDU"
redner_id_mdb["999990150"] = "CDU"
redner_id_mdb["999990141"] = "CSU"
redner_id_mdb["999990193"] = "SPD"
redner_id_mdb["999990093"] = "SPD"
redner_id_mdb["999990120"] = "SPD"
redner_id_mdb["999990129"] = "SPD"
redner_id_mdb["999990145"] = "SPD"
redner_id_mdb["999990144"] = "CDU"
redner_id_mdb["999990125"] = "CDU"
redner_id_mdb["999990147"] = "CSU"
redner_id_mdb["999990146"] = "SPD"
redner_id_mdb["999990121"] = "SPD"
redner_id_mdb["999990148"] = "SPD"
redner_id_mdb["999990122"] = "BÜNDNIS 90/DIE GRÜNEN"
redner_id_mdb["999990123"] = "DIE LINKE"
redner_id_mdb["999990148"] = "SPD"
redner_id_mdb["999990122"] = "BÜNDNIS 90/DIE GRÜNEN"
redner_id_mdb["999990123"] = "DIE LINKE"
redner_id_mdb["999990124"] = "SPD"
redner_id_mdb["999990078"] = "FDP"
redner_id_mdb["999990082"] = "SPD"

In [11]:
# Import der Bundestagsreden via gültigem API-Key für die 5 Betrachtungszeiträume
# API-Key gültig bis zunächst Ende Mai 2027
api = "R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ"

# Eingrenzung Zeitraum Nr. 1, Umwandlung in json-Format & Extraktion der XML-URLs (=Protokolle) aus dem Dictionary
html_zeitraum_1 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2022-01-01&f.datum.end=2022-03-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_1 = json.loads(html_zeitraum_1)
# Erfassen der XML-URLs in einer Liste
data_zeitraum_1_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_1["documents"]
    if "xml_url" in doc["fundstelle"]
]   # 17 Protokolle

# Eingrenzung Zeitraum Nr. 2...
html_zeitraum_2 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2023-10-01&f.datum.end=2023-12-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_2 = json.loads(html_zeitraum_2)
data_zeitraum_2_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_2["documents"]
    if "xml_url" in doc["fundstelle"]
]  # 19 Protokolle

# Eingrenzung Zeitraum Nr. 3...
html_zeitraum_3 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2024-10-01&f.datum.end=2024-12-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_3 = json.loads(html_zeitraum_3)
data_zeitraum_3_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_3["documents"]
    if "xml_url" in doc["fundstelle"]
]   # 19 Protokolle

# Eingrenzung Zeitraum Nr. 4...
html_zeitraum_4 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2025-01-01&f.datum.end=2025-02-23&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_4 = json.loads(html_zeitraum_4)
data_zeitraum_4_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_4["documents"]
    if "xml_url" in doc["fundstelle"]
]    # 4 Protokolle

# Eingrenzung Zeitraum Nr. 5...
html_zeitraum_5 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2025-05-01&f.datum.end=2025-07-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_5 = json.loads(html_zeitraum_5)
data_zeitraum_5_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_5["documents"]
    if "xml_url" in doc["fundstelle"]   
]    # 18 Protokolle

In [12]:
# Anzahl Reden pro Partei und Zeitraum
# zunächst leere Listen
text_liste_CDU_1 = []
text_liste_CDU_2 = []
text_liste_CDU_3 = []
text_liste_CDU_4 = []
text_liste_CDU_5 = []

text_liste_SPD_1 = []
text_liste_SPD_2 = []
text_liste_SPD_3 = []
text_liste_SPD_4 = []
text_liste_SPD_5 = []

text_liste_FDP_1 = []
text_liste_FDP_2 = []
text_liste_FDP_3 = []
text_liste_FDP_4 = []
text_liste_FDP_5 = []

text_liste_Grüne_1 = []
text_liste_Grüne_2 = []
text_liste_Grüne_3 = []
text_liste_Grüne_4 = []
text_liste_Grüne_5 = []

text_liste_Linke_1 = []
text_liste_Linke_2 = []
text_liste_Linke_3 = []
text_liste_Linke_4 = []
text_liste_Linke_5 = []

text_liste_AfD_1 = []
text_liste_AfD_2 = []
text_liste_AfD_3 = []
text_liste_AfD_4 = []
text_liste_AfD_5 = []

# Durchlauf aller extrahierter XML-URLs pro Zeitraum (1 bis 5), Anhängen der Reden pro Partei und Zeitraum an die obigen Listen
for i in data_zeitraum_1_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_1.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_1.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_1.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_1.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_1.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_1.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_1.append(p)
    if redner_id_mdb[id] == "GRÜNE":
      text_liste_Grüne_1.append(p)

for i in data_zeitraum_2_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_2.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_2.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_2.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_2.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_2.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_2.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_2.append(p)
    if redner_id_mdb[id] == "GRÜNE":
      text_liste_Grüne_2.append(p)

for i in data_zeitraum_3_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_3.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_3.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_3.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_3.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_3.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_3.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_3.append(p)
    if redner_id_mdb[id] == "GRÜNE":
      text_liste_Grüne_3.append(p)

for i in data_zeitraum_4_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_4.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_4.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_4.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_4.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_4.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_4.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_4.append(p)
    if redner_id_mdb[id] == "GRÜNE":
      text_liste_Grüne_4.append(p)

for i in data_zeitraum_5_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_5.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_5.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_5.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_5.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_5.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_5.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_5.append(p)
    if redner_id_mdb[id] == "GRÜNE":
      text_liste_Grüne_5.append(p)

In [13]:
# Bereinigung der Texte um Texte der Bundestagspräsidentin und sonstigen Nicht-Rede-Elementen

def bereinige_text_liste(liste):
    bereinigte_liste = []
    for i in liste:
        i = str(i[1:]).replace('<p klasse="J_1">', "")
        i = i.replace('<p klasse="J">', "")
        i = i.replace('<p klasse="O">', "")
        i = i.replace("<p klasse=", "")
        i = re.sub(r'-\s+', '', i)    # neu
        i = re.sub(r'[^\w\s\.!\?€,,:\(\)\-%]', '', i)  # neu
        i = i.replace("p, ", " ")
        i = i.replace(".p, p", ".")
        i = i.replace(":p, ",": ")
        i = re.sub(r".*?hat als Nächstes das Wort für.*?\.", "", i)   # dies sind Texte von der Bundestagspräsidentin und müssen aussortiert werden
        i = re.sub(r'Das\s+Wort\s+hat\s+nun\s+für\s+die\s+Fraktion\s+[\w\s.-]+[\.!?]', '', i)
        i = re.sub(r'Ich\s+erteile\s+das\s+Wort\s+als\s+Nächstes\s+[\w\s. -ÄÖÜäöüß]+[\.!?]', '', i)
        i = re.sub(r'Für\s+die\s+Fraktion\s+[\w\s.-]+\s+hat\s+nun\s+das\s+Wort\s+[\w\s.-]+[\.!?]', '', i)
        i = re.sub(r'Für\s+die\s+Fraktion\s+[\w\s.-]+\s+spricht\s+[\w\s.-]+[\.!?]', '', i)
        i = re.sub(r'Für\s+die\s+[\w\s.-]+\s+hat\s+nun\s+[\w\s.-]+\s+das\s+Wort[\.!?]', '', i)
        i = re.sub(r'^[A-Z][a-zßäöü]+ [A-Z][a-zßäöü]+, kommen Sie bitte zu Ihrer Rede\.', '', i)
        i = re.sub(r"Das Wort für .*? hat .*?\.", "", i)
        i = i.replace("Kommen Sie bitte zum Ende Ihrer Rede.", "")
        i = i.replace("Kommen Sie bitte zum Ende. ", "")
        i = i.replace("Es ist ihre erste Rede.", "")
        i = re.sub(r'rednerredner\s+id\d+[\w\s.-]*:', '', i)
        i = i.replace(", kommen Sie bitte zum Ende Ihrer Rede.p", "")
        i = i.replace(", kommen Sie bitte zum Schluss.", "")
        i = i.replace(", kommen Sie bitte zu Ihrer Rede.", "")
        i = i.replace("Sehr geehrter Herr Botschafter, ", "")
        i = re.sub(r'Die\s+nächste\s+Rednerin\s+ist\s+[\w\s.-]+\s+für\s+die\s+[\w\s.-]+[\.!?]', '', i)
        i = re.sub(r'Der\s+nächste\s+Redner\s+ist\s+[\w\s.-]+\s+für\s+die\s+[\w\s.-]+[\.!?]', '', i)
        i = re.sub(r'[D|d]er nächste Redner in der Debatte ist [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)? [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)?\.?', '', i)
        i = re.sub(r'[D|d]ie nächste Rednerin in der Debatte ist [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)? [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)?\.?', '', i)
        i = re.sub(r'rednerredner\s+[\w\s. -ÄÖÜäöüß]+:', '', i)
        i = re.sub(r'[F|f]ür die [A-Za-zßäöüÄÖÜ\s\-\/0-9]+-Fraktion erhält das Wort [A-Z][a-zßäöüÄÖÜ]+(?:-[A-Z][a-zßäöüÄÖÜ]+)?(?: [A-Z][a-zßäöüÄÖÜ]+(?:-[A-Z][a-zßäöüÄÖÜ]+)?)?\.?', '', i)
        i = re.sub(r'Als\s+Nächstes\s+spricht\s+[\w\s. -ÄÖÜäöüß]+[\.!?]', '', i)
        i = re.sub(r'[I|i]ch darf für die Fraktion (?:[A-Zßäöüa-z\s\/äöü]+) [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)? [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)? aufrufen\.?', '', i)
        i = re.sub(r'[I|i]ch erteile das Wort für die nächste Rede (?:[A-ZßäöüÄÖÜa-z\s\/äöü]+) [A-Z][a-zßäöüÄÖÜ]+(?:-[A-Z][a-zßäöüÄÖÜ]+)?(?: [A-Z][a-zßäöüÄÖÜ]+(?:-[A-Z][a-zßäöüÄÖÜ]+)?)?\.?', '', i)
        i = re.sub(r'[I|i]ch erteile als Nächstes das Wort de[mr] Abgeordneten [A-Z][a-zßäöüÄÖÜ]+(?: [A-Z][a-zßäöüÄÖÜ]+)* für die [A-Za-zßäöüÄÖÜ\s\/]+(?:\.)?', '', i)
        i = re.sub(r'[D|d]ie nächste Rede hält [A-Z][a-zßäöüÄÖÜ]+(?: [A-Z][a-zßäöüÄÖÜ]+)* für die [A-Za-zßäöüÄÖÜ\s\-\/0-9]+(?:\.)?', '', i)
        i = re.sub(r'[I|i]ch darf aufrufen für die [A-Za-zßäöüÄÖÜ\s\-\/0-9]+(?:\.)?', '', i)
        i = re.sub(r'[D|d]ie nächste Rede hält [A-Z][a-zßäöüÄÖÜ]+(?: [A-Z][a-zßäöüÄÖÜ]+)*(?:\.)?', '', i)
        i = re.sub(r'[A[Aa]ls nächste(?:r)? (?:Rednerin|Redner) hat [A-ZßäöüÄÖÜa-z\s\-\/\.]+ das Wort\.?', '', i)
        i = re.sub(r'[E[Ee]benfalls zur ersten Rede erteile ich das Wort [A-ZßäöüÄÖÜa-z\s\-\/\.]+\.?', '', i)
        i = re.sub(r'[Ii]ch darf [A-ZßäöüÄÖÜa-z\s\-\/\.]+ das Wort erteilen\.?', '', i)
        i = re.sub(r'[Ii]ch darf [A-ZßäöüÄÖÜa-z\s\-\/\.]+ aufrufen\.?', '', i)
        i = re.sub(r'[Ii]ch erteile das Wort [A-ZßäöüÄÖÜa-z\s\-\/\.]+\.?', '', i)
        i = re.sub(r'[Vv]ielen Dank und Gratulation zu Ihrer ersten Rede, (?:Frau|Herr) [A-ZßäöüÄÖÜa-z\s\-\/\.]+(?:\.)?', '', i)
        i = re.sub(r'[Ff]ür die [A-Za-zßäöüÄÖÜ\s\-\/0-9]+ das Wort zu seiner ersten Rede\.', '', i)
        i = re.sub(r'[Ff]ür die [A-Za-zßäöüÄÖÜ\s\-\/0-9]+ das Wort zu ihrer ersten Rede\.', '', i)
        i = re.sub(r'[Dd]er nächste Redner in der Debatte: für [A-Za-zßäöüÄÖÜ\s\-\/0-9]+(?:\.)?', '', i)
        i = re.sub(r'[Dd]ie nächste Rednerin in der Debatte: für [A-Za-zßäöüÄÖÜ\s\-\/0-9]+(?:\.)?', '', i)
        i = re.sub(r'[Dd]ann rufe ich [A-ZßäöüÄÖÜa-z\s\-\/\.]+ in der Debatte auf: [A-Z][a-zßäöüÄÖÜ]+(?:-[A-Z][a-zßäöüÄÖÜ]+)?(?: [A-Z][a-zßäöüÄÖÜ]+)? für [A-Za-zßäöüÄÖÜ\s\-\/0-9]+(?:\.)?', '', i)
        i = re.sub(r'[Zz]u (?:seiner|ihrer) ersten Rede hat nun [A-ZßäöüÄÖÜa-z\s\-\/\.]+ das Wort\.?', '', i)
        i = re.sub(r'von der [A-Za-zßäöüÄÖÜ\s\-\/0-9]+, für (?:den|die) es hier die erste Rede ist\.?', '', i)
        i = i.replace("Vielen Dank Ihnen. ", "")
        i = i.removesuffix('p')
        i = re.sub(r"\s+", " ", i)
        i = i.replace("\xa0", "")
        i = i.replace("</p>", "")
        i = i.replace("</p", "")
        i = i.replace("p, p, ", "")
        i = i.lstrip()
        i = i.rstrip(" ")
        bereinigte_liste.append(i)
    return bereinigte_liste

# Zeitraum 1
text_liste_CDU_1_clean = bereinige_text_liste(text_liste_CDU_1)
text_liste_SPD_1_clean = bereinige_text_liste(text_liste_SPD_1)
text_liste_Linke_1_clean = bereinige_text_liste(text_liste_Linke_1)
text_liste_AfD_1_clean = bereinige_text_liste(text_liste_AfD_1)
text_liste_Grüne_1_clean = bereinige_text_liste(text_liste_Grüne_1)
text_liste_FDP_1_clean = bereinige_text_liste(text_liste_FDP_1)

# Zeitraum 2
text_liste_CDU_2_clean = bereinige_text_liste(text_liste_CDU_2)
text_liste_SPD_2_clean = bereinige_text_liste(text_liste_SPD_2)
text_liste_Linke_2_clean = bereinige_text_liste(text_liste_Linke_2)
text_liste_AfD_2_clean = bereinige_text_liste(text_liste_AfD_2)
text_liste_Grüne_2_clean = bereinige_text_liste(text_liste_Grüne_2)
text_liste_FDP_2_clean = bereinige_text_liste(text_liste_FDP_2)

# Zeitraum 3
text_liste_CDU_3_clean = bereinige_text_liste(text_liste_CDU_3)
text_liste_SPD_3_clean = bereinige_text_liste(text_liste_SPD_3)
text_liste_Linke_3_clean = bereinige_text_liste(text_liste_Linke_3)
text_liste_AfD_3_clean = bereinige_text_liste(text_liste_AfD_3)
text_liste_Grüne_3_clean = bereinige_text_liste(text_liste_Grüne_3)
text_liste_FDP_3_clean = bereinige_text_liste(text_liste_FDP_3)

# Zeitraum 4
text_liste_CDU_4_clean = bereinige_text_liste(text_liste_CDU_4)
text_liste_SPD_4_clean = bereinige_text_liste(text_liste_SPD_4)
text_liste_Linke_4_clean = bereinige_text_liste(text_liste_Linke_4)
text_liste_AfD_4_clean = bereinige_text_liste(text_liste_AfD_4)
text_liste_Grüne_4_clean = bereinige_text_liste(text_liste_Grüne_4)
text_liste_FDP_4_clean = bereinige_text_liste(text_liste_FDP_4)

# Zeitraum 5
text_liste_CDU_5_clean = bereinige_text_liste(text_liste_CDU_5)
text_liste_SPD_5_clean = bereinige_text_liste(text_liste_SPD_5)
text_liste_Linke_5_clean = bereinige_text_liste(text_liste_Linke_5)
text_liste_AfD_5_clean = bereinige_text_liste(text_liste_AfD_5)
text_liste_Grüne_5_clean = bereinige_text_liste(text_liste_Grüne_5)
text_liste_FDP_5_clean = bereinige_text_liste(text_liste_FDP_5)

In [14]:
# aus jeder Liste einen großen String pro Partei und Betrachtungszeitraum
text_liste_CDU_1_str = "".join(text_liste_CDU_1_clean)
text_liste_SPD_1_str = "".join(text_liste_SPD_1_clean)
text_liste_AfD_1_str = "".join(text_liste_AfD_1_clean)
text_liste_Linke_1_str = "".join(text_liste_Linke_1_clean)
text_liste_Grüne_1_str = "".join(text_liste_Grüne_1_clean)
text_liste_FDP_1_str = "".join(text_liste_FDP_1_clean)

text_liste_CDU_2_str = "".join(text_liste_CDU_2_clean)
text_liste_SPD_2_str = "".join(text_liste_SPD_2_clean)
text_liste_AfD_2_str = "".join(text_liste_AfD_2_clean)
text_liste_Linke_2_str = "".join(text_liste_Linke_2_clean)
text_liste_Grüne_2_str = "".join(text_liste_Grüne_2_clean)
text_liste_FDP_2_str = "".join(text_liste_FDP_2_clean)

text_liste_CDU_3_str = "".join(text_liste_CDU_3_clean)
text_liste_SPD_3_str = "".join(text_liste_SPD_3_clean)
text_liste_AfD_3_str = "".join(text_liste_AfD_3_clean)
text_liste_Linke_3_str = "".join(text_liste_Linke_3_clean)
text_liste_Grüne_3_str = "".join(text_liste_Grüne_3_clean)
text_liste_FDP_3_str = "".join(text_liste_FDP_3_clean)

text_liste_CDU_4_str = "".join(text_liste_CDU_4_clean)
text_liste_SPD_4_str = "".join(text_liste_SPD_4_clean)
text_liste_AfD_4_str = "".join(text_liste_AfD_4_clean)
text_liste_Linke_4_str = "".join(text_liste_Linke_4_clean)
text_liste_Grüne_4_str = "".join(text_liste_Grüne_4_clean)
text_liste_FDP_4_str = "".join(text_liste_FDP_4_clean)

text_liste_CDU_5_str = "".join(text_liste_CDU_5_clean)
text_liste_SPD_5_str = "".join(text_liste_SPD_5_clean)
text_liste_AfD_5_str = "".join(text_liste_AfD_5_clean)
text_liste_Linke_5_str = "".join(text_liste_Linke_5_clean)
text_liste_Grüne_5_str = "".join(text_liste_Grüne_5_clean)
text_liste_FDP_5_str = "".join(text_liste_FDP_5_clean)

In [15]:
# Ermittlung der Werte für die beiden Indizes pro Partei und Betrachtungszeitraum
# Lesbarkeit Periode 1
CDU_WSF_Rede_1 = textstat.wiener_sachtextformel(text_liste_CDU_1_str, 1)
SPD_WSF_Rede_1 = textstat.wiener_sachtextformel(text_liste_SPD_1_str, 1)
Linke_WSF_Rede_1 = textstat.wiener_sachtextformel(text_liste_Linke_1_str, 1)
AfD_WSF_Rede_1 = textstat.wiener_sachtextformel(text_liste_AfD_1_str, 1)
Grüne_WSF_Rede_1 = textstat.wiener_sachtextformel(text_liste_Grüne_1_str, 1)
FDP_WSF_Rede_1 = textstat.wiener_sachtextformel(text_liste_FDP_1_str, 1)

CDU_FRE_Rede_1 = textstat.flesch_reading_ease(text_liste_CDU_1_str)
SPD_FRE_Rede_1 = textstat.flesch_reading_ease(text_liste_SPD_1_str)
Linke_FRE_Rede_1 = textstat.flesch_reading_ease(text_liste_Linke_1_str)
AfD_FRE_Rede_1 = textstat.flesch_reading_ease(text_liste_AfD_1_str)
Grüne_FRE_Rede_1 = textstat.flesch_reading_ease(text_liste_Grüne_1_str)
FDP_FRE_Rede_1 = textstat.flesch_reading_ease(text_liste_FDP_1_str)

# Lesbarkeit Periode 2
CDU_WSF_Rede_2 = textstat.wiener_sachtextformel(text_liste_CDU_2_str, 1)
SPD_WSF_Rede_2 = textstat.wiener_sachtextformel(text_liste_SPD_2_str, 1)
Linke_WSF_Rede_2 = textstat.wiener_sachtextformel(text_liste_Linke_2_str, 1)
AfD_WSF_Rede_2 = textstat.wiener_sachtextformel(text_liste_AfD_2_str, 1)
Grüne_WSF_Rede_2 = textstat.wiener_sachtextformel(text_liste_Grüne_2_str, 1)
FDP_WSF_Rede_2 = textstat.wiener_sachtextformel(text_liste_FDP_2_str, 1)

CDU_FRE_Rede_2 = textstat.flesch_reading_ease(text_liste_CDU_2_str)
SPD_FRE_Rede_2 = textstat.flesch_reading_ease(text_liste_SPD_2_str)
Linke_FRE_Rede_2 = textstat.flesch_reading_ease(text_liste_Linke_2_str)
AfD_FRE_Rede_2 = textstat.flesch_reading_ease(text_liste_AfD_2_str)
Grüne_FRE_Rede_2 = textstat.flesch_reading_ease(text_liste_Grüne_2_str)
FDP_FRE_Rede_2 = textstat.flesch_reading_ease(text_liste_FDP_2_str)

# Lesbarkeit Periode 3
CDU_WSF_Rede_3 = textstat.wiener_sachtextformel(text_liste_CDU_3_str, 1)
SPD_WSF_Rede_3 = textstat.wiener_sachtextformel(text_liste_SPD_3_str, 1)
Linke_WSF_Rede_3 = textstat.wiener_sachtextformel(text_liste_Linke_3_str, 1)
AfD_WSF_Rede_3 = textstat.wiener_sachtextformel(text_liste_AfD_3_str, 1)
Grüne_WSF_Rede_3 = textstat.wiener_sachtextformel(text_liste_Grüne_3_str, 1)
FDP_WSF_Rede_3 = textstat.wiener_sachtextformel(text_liste_FDP_3_str, 1)

CDU_FRE_Rede_3 = textstat.flesch_reading_ease(text_liste_CDU_3_str)
SPD_FRE_Rede_3 = textstat.flesch_reading_ease(text_liste_SPD_3_str)
Linke_FRE_Rede_3 = textstat.flesch_reading_ease(text_liste_Linke_3_str)
AfD_FRE_Rede_3 = textstat.flesch_reading_ease(text_liste_AfD_3_str)
Grüne_FRE_Rede_3 = textstat.flesch_reading_ease(text_liste_Grüne_3_str)
FDP_FRE_Rede_3 = textstat.flesch_reading_ease(text_liste_FDP_3_str)

# Lesbarkeit Periode 4
CDU_WSF_Rede_4 = textstat.wiener_sachtextformel(text_liste_CDU_4_str, 1)
SPD_WSF_Rede_4 = textstat.wiener_sachtextformel(text_liste_SPD_4_str, 1)
Linke_WSF_Rede_4 = textstat.wiener_sachtextformel(text_liste_Linke_4_str, 1)
AfD_WSF_Rede_4 = textstat.wiener_sachtextformel(text_liste_AfD_4_str, 1)
Grüne_WSF_Rede_4 = textstat.wiener_sachtextformel(text_liste_Grüne_4_str, 1)
FDP_WSF_Rede_4 = textstat.wiener_sachtextformel(text_liste_FDP_4_str, 1)

CDU_FRE_Rede_4 = textstat.flesch_reading_ease(text_liste_CDU_4_str)
SPD_FRE_Rede_4 = textstat.flesch_reading_ease(text_liste_SPD_4_str)
Linke_FRE_Rede_4 = textstat.flesch_reading_ease(text_liste_Linke_4_str)
AfD_FRE_Rede_4 = textstat.flesch_reading_ease(text_liste_AfD_4_str)
Grüne_FRE_Rede_4 = textstat.flesch_reading_ease(text_liste_Grüne_4_str)
FDP_FRE_Rede_4 = textstat.flesch_reading_ease(text_liste_FDP_4_str)

# Lesbarkeit Periode 5
CDU_WSF_Rede_5 = textstat.wiener_sachtextformel(text_liste_CDU_5_str, 1)
SPD_WSF_Rede_5 = textstat.wiener_sachtextformel(text_liste_SPD_5_str, 1)
Linke_WSF_Rede_5 = textstat.wiener_sachtextformel(text_liste_Linke_5_str, 1)
AfD_WSF_Rede_5 = textstat.wiener_sachtextformel(text_liste_AfD_5_str, 1)
Grüne_WSF_Rede_5 = textstat.wiener_sachtextformel(text_liste_Grüne_5_str, 1)
FDP_WSF_Rede_5 = textstat.wiener_sachtextformel(text_liste_FDP_5_str, 1)

CDU_FRE_Rede_5 = textstat.flesch_reading_ease(text_liste_CDU_5_str)
SPD_FRE_Rede_5 = textstat.flesch_reading_ease(text_liste_SPD_5_str)
Linke_FRE_Rede_5 = textstat.flesch_reading_ease(text_liste_Linke_5_str)
AfD_FRE_Rede_5 = textstat.flesch_reading_ease(text_liste_AfD_5_str)
Grüne_FRE_Rede_5 = textstat.flesch_reading_ease(text_liste_Grüne_5_str)
FDP_FRE_Rede_5 = textstat.flesch_reading_ease(text_liste_FDP_5_str)

In [16]:
# PandaFrame-Erstellung
df_reden = pd.DataFrame({
    ("Periode 1", "1. WSF"): [CDU_WSF_Rede_1, SPD_WSF_Rede_1, Linke_WSF_Rede_1, AfD_WSF_Rede_1, Grüne_WSF_Rede_1, FDP_WSF_Rede_1],
    ("Periode 1", "Flesch-Amstad"): [CDU_FRE_Rede_1, SPD_FRE_Rede_1, Linke_FRE_Rede_1, AfD_FRE_Rede_1, Grüne_FRE_Rede_1, FDP_FRE_Rede_1],
    ("Periode 2", "1. WSF"): [CDU_WSF_Rede_2, SPD_WSF_Rede_2, Linke_WSF_Rede_2, AfD_WSF_Rede_2, Grüne_WSF_Rede_2, FDP_WSF_Rede_2],
    ("Periode 2", "Flesch-Amstad"): [CDU_FRE_Rede_2, SPD_FRE_Rede_2, Linke_FRE_Rede_2, AfD_FRE_Rede_2, Grüne_FRE_Rede_2, FDP_FRE_Rede_2],
    ("Periode 3", "1. WSF"): [CDU_WSF_Rede_3, SPD_WSF_Rede_3, Linke_WSF_Rede_3, AfD_WSF_Rede_3, Grüne_WSF_Rede_3, FDP_WSF_Rede_3],
    ("Periode 3", "Flesch-Amstad"): [CDU_FRE_Rede_3, SPD_FRE_Rede_3, Linke_FRE_Rede_3, AfD_FRE_Rede_3, Grüne_FRE_Rede_3, FDP_FRE_Rede_3],
    ("Periode 4", "1. WSF"): [CDU_WSF_Rede_4, SPD_WSF_Rede_4, Linke_WSF_Rede_4, AfD_WSF_Rede_4, Grüne_WSF_Rede_4, FDP_WSF_Rede_4],
    ("Periode 4", "Flesch-Amstad"): [CDU_FRE_Rede_4, SPD_FRE_Rede_4, Linke_FRE_Rede_4, AfD_FRE_Rede_4, Grüne_FRE_Rede_4, FDP_FRE_Rede_4],
    ("Periode 5", "1. WSF"): [CDU_WSF_Rede_5, SPD_WSF_Rede_5, Linke_WSF_Rede_5, AfD_WSF_Rede_5, Grüne_WSF_Rede_5, FDP_WSF_Rede_5],
    ("Periode 5", "Flesch-Amstad"): [CDU_FRE_Rede_5, SPD_FRE_Rede_5, Linke_FRE_Rede_5, AfD_FRE_Rede_5, Grüne_FRE_Rede_5, FDP_FRE_Rede_5],
}).round(1)

df_reden.index = ["CDU/CSU", "SPD", "Linke", "AfD", "Grüne", "FDP"]
df_reden = df_reden.replace(0.0, "-")
df_reden

Periode 1               Periode 2               Periode 3  \
           1. WSF Flesch-Amstad    1. WSF Flesch-Amstad    1. WSF   
CDU/CSU       8.6          50.6       8.6          51.7       8.4   
SPD           9.0          48.8       8.7          50.8       8.6   
Linke         8.5          51.8       9.0          49.2       8.5   
AfD           8.9          50.2       8.7          51.2       8.6   
Grüne         8.9          49.4       8.7          50.7       8.7   
FDP           9.0          48.9       8.8          50.3       8.8   

                      Periode 4               Periode 5                
        Flesch-Amstad    1. WSF Flesch-Amstad    1. WSF Flesch-Amstad  
CDU/CSU          52.5       8.5          52.3       8.9          49.8  
SPD              51.8       8.1          53.7       8.8          50.4  
Linke            51.8       7.9          55.3       8.4          52.8  
AfD              51.9       8.2          54.1       8.9          50.5  
Grüne            51.0       7.9          55.2       8.2          53.4  
FDP              50.9       9.3          48.2         -             -

In [17]:
# in LaTex überführen

latex_code = df_reden.to_latex(index=False, caption='Reden', label='tab:meine_tabelle')
with open('tabelle_reden_wstf_fre.tex', 'w') as f:
    f.write(latex_code)

**3. Teil: Leichte Wahlprogramme**

In [18]:
# Einlesen der Leichten Programme

# für CDU/CSU
with open("CDU_leicht.txt", "r", encoding="utf-8") as datei:
    CDU_text = datei.read()
CDU_text = re.sub(r'-\s+', '', CDU_text)                               # Ersetzt den Bindestrich gefolgt von einem oder mehreren Leerzeichen durch nichts
CDU_text = re.sub(r'[^\w\s\.!\?€,,:\(\)\-%]', '', CDU_text)            # Sonderzeichen-Filter, löscht alles, was nicht ausdrücklich erhalten bleiben soll
CDU_text = re.sub(r'([A-Za-zÄÖÜäöüß])\1{3,}', r'\1\1\1', CDU_text)     # Begrenzung von extremen Wiederholungen
CDU_text = re.sub(r'\s+', ' ', CDU_text).strip()                       # Normalisierung von Whitespaces (ersetzt durch ein Leerzeichen)

# für SPD
with open("SPD_leicht.txt", "r", encoding="utf-8") as datei:
    SPD_text = datei.read()
SPD_text = re.sub(r'-\s+', '', SPD_text)
SPD_text = re.sub(r'[^\w\s\.!\?€,,:\(\)\-%]', '', SPD_text)
SPD_text = re.sub(r'([A-Za-zÄÖÜäöüß])\1{3,}', r'\1\1\1', SPD_text)
SPD_text = re.sub(r'\s+', ' ', SPD_text).strip()

# für Die Linke
with open("Linke_leicht.txt", "r", encoding="utf-8") as datei:
    Linke_text = datei.read()
Linke_text = re.sub(r'-\s+', '', Linke_text)
Linke_text = re.sub(r'[^\w\s\.!\?€,,:\(\)\-%]', '', Linke_text)
Linke_text = re.sub(r'([A-Za-zÄÖÜäöüß])\1{3,}', r'\1\1\1', Linke_text)
Linke_text = re.sub(r'\s+', ' ', Linke_text).strip()

# für Bündnis 90/Grüne
with open("Grüne_leicht.txt", "r", encoding="utf-8") as datei:
    Grüne_text = datei.read()
Grüne_text = re.sub(r'-\s+', '', Grüne_text)
Grüne_text = re.sub(r'[^\w\s\.!\?€,,:\(\)\-%]', '', Grüne_text)
Grüne_text = re.sub(r'([A-Za-zÄÖÜäöüß])\1{3,}', r'\1\1\1', Grüne_text)
Grüne_text = re.sub(r'\s+', ' ', Grüne_text).strip()

# für FDP
with open("FDP_leicht.txt", "r", encoding="utf-8") as datei:
    FDP_text = datei.read()
FDP_text = re.sub(r'-\s+', '', FDP_text)
FDP_text = re.sub(r'[^\w\s\.!\?€,,:\(\)\-%]', '', FDP_text)
FDP_text = re.sub(r'([A-Za-zÄÖÜäöüß])\1{3,}', r'\1\1\1', FDP_text)
FDP_text = re.sub(r'\s+', ' ', FDP_text).strip()

In [19]:
# Bereinigen der Texte => Umformen von Abkürzungen in Text um Eindruck von einem Satzende zu vermeiden

def bereinige_abkuerzungen(text):
    # Dictionary mit den Top 20 Ersetzungen (Regex-Muster als Key)
    ersetzungen = {
        r'\bz\.\s*B\.': 'zum Beispiel',   # r'\bz\.\s*B\.\b': 'zum Beispiel',
        r'\bu\.\s*a\.': 'unter anderem',
        r'\bd\.\s*h\.': 'das heißt',
        r'\bbzw\.': 'beziehungsweise',
        r'\bbzw': 'beziehungsweise',   # Variante ohne Punkt
        r'\bsog\.': 'sogenannte',
        r'\bca\.': 'circa',
        r'\bevtl\.': 'eventuell',
        r'\binkl\.': 'inklusive',
        r'\betc\.': 'et cetera',
        r'\bvgl\.': 'vergleiche',
        r'\bs\.': 'siehe',
        r'\bu\.\s*v\.\s*m\.': 'und vieles mehr',
        r'\bu\.\s*ä\.': 'und ähnliche',
        r'\bggf\.': 'gegebenenfalls',
        r'\bzzgl\.': 'zuzüglich',
        r'\bebd\.': 'ebenda',
        r'\bo\.\s*g\.': 'oben genannte',
        r'\bu\.\s*g\.': 'unten genannte',
        r'\bi\.\s*d\.\s*R\.': 'in der Regel',
        r'\bv\.\s*a\.': 'vor allem',

        # 2. Titel & Personen (Verhindern Satzabbruch mitten im Fluss)
        r'\bDr\.': 'Doktor',
        r'\bProf\.': 'Professor',
        r'\bFr\.': 'Frau',
        r'\bHr\.': 'Herr',

        # 3. Wortlaengen- & Silben-Verfaelscher
        r'\bbspw\.': 'beispielsweise',
        r'\bbspw': 'beispielsweise',   # Variante ohne Punkt
        r'\bbsp\.': 'Beispiel',
        r'\bJh\.': 'Jahrhundert',
        r'\bMio\.': 'Millionen',
        r'\bMrd\.': 'Milliarden',
        r'\bbetr\.': 'betreffend',
        r'\bbezgl\.': 'bezüglich',
        r'\bvs\.': 'versus'
    }

    # Text Schritt für Schritt bereinigen
    for muster, ersetzung in ersetzungen.items():
        # flags=re.IGNORECASE sorgt dafür, dass auch "Z.B." oder "Bzw." gefunden werden
        text = re.sub(muster, ersetzung, text, flags=re.IGNORECASE)

    return text

CDU_text = bereinige_abkuerzungen(CDU_text)
SPD_text = bereinige_abkuerzungen(SPD_text)
Grüne_text = bereinige_abkuerzungen(Grüne_text)
Linke_text = bereinige_abkuerzungen(Linke_text)
FDP_text = bereinige_abkuerzungen(FDP_text)

In [20]:
# Ermittlung der Werte für die beiden Indizes
CDU_WSF = textstat.wiener_sachtextformel(CDU_text, 1)
SPD_WSF = textstat.wiener_sachtextformel(SPD_text, 1)
Linke_WSF = textstat.wiener_sachtextformel(Linke_text, 1)
Grüne_WSF = textstat.wiener_sachtextformel(Grüne_text, 1)
FDP_WSF = textstat.wiener_sachtextformel(FDP_text, 1)

CDU_FRE = textstat.flesch_reading_ease(CDU_text)
SPD_FRE = textstat.flesch_reading_ease(SPD_text)
Linke_FRE = textstat.flesch_reading_ease(Linke_text)
Grüne_FRE = textstat.flesch_reading_ease(Grüne_text)
FDP_FRE = textstat.flesch_reading_ease(FDP_text)

In [21]:
# Erstellung des PandasDataFrame
wsf_series = pd.Series({
    "CDU": CDU_WSF,
    "SPD": SPD_WSF,
    "Linke": Linke_WSF,
    "Grüne": Grüne_WSF,
    "FDP": FDP_WSF}).round(1)

FRE_series = pd.Series({
    "CDU": CDU_FRE,
    "SPD": SPD_FRE,
    "Linke": Linke_FRE,
    "Grüne": Grüne_FRE,
    "FDP": FDP_FRE}).round(1)

df_wahlprogramm = pd.DataFrame(
    [wsf_series, FRE_series],
    index=["1. Wiener Sachtextformel", "Flesch-Amstad-Formel"]
)

df_wahlprogramm.columns = ["CDU/CSU", "SPD", "Linke", "Grüne", "FDP"]
df_wahlprogramm["Durchschnitt"] = df_wahlprogramm.mean(axis=1).round(1)
df_wahlprogramm

,CDU/CSU,SPD,Linke,Grüne,FDP,Durchschnitt
1. Wiener Sachtextformel,6.8,5.6,5.0,6.7,6.0,6.0
Flesch-Amstad-Formel,64.4,69.9,70.9,62.7,67.8,67.1


In [22]:
# in LaTex überführen

latex_code = df_wahlprogramm.to_latex(index=False, caption='Meine Übersicht', label='tab:meine_tabelle')
with open('tabelle_leicht_wstf_fre.tex', 'w') as f:
    f.write(latex_code)